# TARDIS

In [64]:
import pandas as pd
import numpy as np

In [65]:
df = pd.read_csv("dataset.csv", sep=",")
print("(Nb rows, Nb columns): ", df.shape)
df.info()

(Nb rows, Nb columns):  (40, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               40 non-null     object 
 1   departure_station  39 non-null     object 
 2   arrival_station    39 non-null     object 
 3   scheduled_time     37 non-null     object 
 4   actual_time        36 non-null     object 
 5   delay_minutes      30 non-null     float64
dtypes: float64(1), object(5)
memory usage: 2.0+ KB


In [66]:
nb_duplicates = df.duplicated().sum()

if nb_duplicates > 0:
    df = df.drop_duplicates()

print("Nb duplicates: ", nb_duplicates)
print("(Nb rows, Nb columns): ", df.shape)

Nb duplicates:  0
(Nb rows, Nb columns):  (40, 6)


In [67]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["departure_station"] = df["departure_station"].fillna("Unknown")
df["arrival_station"] = df["arrival_station"].fillna("Unknown")

scheduled_time = df["scheduled_time"].astype(str)
scheduled_time_fixed = scheduled_time.str.replace("-", ":")
df["scheduled_time"] = pd.to_datetime(
    scheduled_time_fixed, format="%H:%M", errors="coerce"
).dt.time

actual_time = df["actual_time"].astype(str)
actual_time_fixed = actual_time.str.replace("-", ":")
df["actual_time"] = pd.to_datetime(
    actual_time_fixed, format="%H:%M", errors="coerce"
).dt.time

df["delay_minutes"] = pd.to_numeric(df["delay_minutes"], errors="coerce")
for i in df.index:
    value = df.loc[i, "delay_minutes"]
    if pd.notnull(value) and value < 0:
        df.loc[i, "delay_minutes"] = np.nan

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               14 non-null     datetime64[ns]
 1   departure_station  40 non-null     object        
 2   arrival_station    40 non-null     object        
 3   scheduled_time     24 non-null     object        
 4   actual_time        18 non-null     object        
 5   delay_minutes      29 non-null     float64       
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 2.0+ KB


## Data Cleaning

In [68]:
print(df.isnull().sum())

date                 26
departure_station     0
arrival_station       0
scheduled_time       16
actual_time          22
delay_minutes        11
dtype: int64


In [69]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [70]:
df["departure_station"] = df["departure_station"].fillna("Unknown")
df["arrival_station"] = df["arrival_station"].fillna("Unknown")

In [71]:
df["delay_minutes"] = pd.to_numeric(df["delay_minutes"], errors="coerce")
df.loc[df["delay_minutes"] < 0, "delay_minutes"] = np.nan

In [72]:
df = df.dropna(subset=["date", "scheduled_time", "actual_time", "delay_minutes"])
print(df.shape)

(10, 6)


## Feature Engineering

In [73]:
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month

In [74]:
df["scheduled_hour"] = df["scheduled_time"].apply(
    lambda x: x.hour if pd.notnull(x) else np.nan
)

In [75]:
df["is_peak_hour"] = df["scheduled_hour"].apply(
    lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0
)

In [76]:
df["delay_category"] = pd.cut(
    df["delay_minutes"],
    bins=[-1, 5, 15, 30, float("inf")],
    labels=["On time / slight", "Short delay", "Medium delay", "Long delay"]
)

In [77]:
df.head()
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 0 to 37
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               10 non-null     datetime64[ns]
 1   departure_station  10 non-null     object        
 2   arrival_station    10 non-null     object        
 3   scheduled_time     10 non-null     object        
 4   actual_time        10 non-null     object        
 5   delay_minutes      10 non-null     float64       
 6   day_of_week        10 non-null     object        
 7   month              10 non-null     int32         
 8   scheduled_hour     10 non-null     int64         
 9   is_peak_hour       10 non-null     int64         
 10  delay_category     10 non-null     category      
dtypes: category(1), datetime64[ns](1), float64(1), int32(1), int64(2), object(5)
memory usage: 1.0+ KB
None
date                 0
departure_station    0
arrival_station      0
scheduled

In [79]:
df.to_csv("cleaned_dataset.csv", index=False)